# How much of a forecasting foundation model is actually load-bearing?

**The question.** Chronos-Bolt is a deployed, widely-used time-series foundation model.
If you delete a randomly chosen half of its feed-forward features, how much worse does it
forecast?

The answer on the 9M checkpoint was: **essentially not at all.** This notebook asks
whether that survives scale, across the full ladder from 9M to 205M parameters
(8,192 -> 73,728 FFN features), and on Chronos-2, the current state of the art.

### Why this is interesting rather than just an ablation study

Every result here is *causal*, not correlational -- we delete a part and measure what
breaks, using **Weighted Quantile Loss**, the metric Chronos itself reports. And there is a
prediction worth committing to before running:

> If the redundancy is just an artifact of a small over-parameterised model, the removable
> fraction should **shrink** as models get bigger. If it **grows**, redundancy is a
> property of how these models store forecasting behaviour.

Early evidence on CPU (tiny vs mini) points the second way -- removing 50% of `mini`
*improved* WQL by 2.8%. This notebook settles it on real hardware.

### The one cell you must not skip

Cell 5 is the **positive control**: zeroing *all* FFN features must wreck the forecast. If
it doesn't, the ablation is silently a no-op -- wrong module path, a copy instead of a
view, grads still attached -- and every number afterwards is noise that looks like a
finding. It runs first, for every model, and asserts.

**Hardware.** Inference only, no training. A100 recommended; the full ladder is ~30-50 min.
Runs on CPU for the two smallest models.

## 1 - Setup

In [ ]:
# transformers >=4.48 breaks chronos on some Python versions
!pip install -q "transformers>=4.40,<4.48" chronos-forecasting certifi
!git clone -q https://github.com/thebnbrkr/marv-titan.git /content/marv-titan 2>/dev/null || true
import sys; sys.path.insert(0, '/content/marv-titan/experiments')

import torch, numpy as np, time, json, os
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")

In [ ]:
# shared helpers live in the repo so this notebook and the script cannot drift apart
from chronos_ablation import LEVELS, load_ett, make_windows, wql, naive_wql
from chronos_scaling import build, scorer, ablate, ablate_all, LADDER

print("ladder:")
for m in LADDER: print("  ", m)
print("\nquantile levels:", np.round(LEVELS,1))

## 2 - Data

ETT = electricity transformer sensors, 7 channels, a standard forecasting benchmark.
Non-overlapping windows: 512 observations of context, 64 to predict.

**WQL** is the metric Chronos reports. Lower is better. The naive last-value forecast is
the floor a model must beat to be doing anything at all.

In [ ]:
DATASET = "ETTh1"          # try ETTm1 (15-min) and ETTh2 as well -- they differ a lot
arr = load_ett(DATASET)
ins, tgt = make_windows(arr, 512, 64)
print(f"{DATASET}: {arr.shape[0]} timesteps x {arr.shape[1]} channels")
print(f"eval set : {len(ins)} windows")
print(f"naive last-value WQL: {naive_wql(ins, tgt, 64):.4f}   <- the floor to beat")

## 3 - One model, in detail

Before the ladder, run a single model end to end so you can see each step. The order
matters: **control first**, then results.

In [ ]:
MODEL = "amazon/chronos-bolt-tiny"

pipe, ffn, names, n_feat = build(MODEL, DEVICE)
ins_d = [x.to(DEVICE) for x in ins]
score = scorer(pipe, ins_d, tgt, 64)
TOTAL = len(ffn) * n_feat

base = score()
print(f"{MODEL}")
print(f"  {len(ffn)} FFN layers x {n_feat} features = {TOTAL}")
print(f"  baseline WQL {base:.4f}   (naive {naive_wql(ins, tgt, 64):.4f})")

# ---- POSITIVE CONTROL -------------------------------------------------------
allz = ablate_all(ffn, score)
print(f"\n  CONTROL: all {TOTAL} FFN features zeroed -> WQL {allz:.4f} ({(allz-base)/base*100:+.0f}%)")
assert allz > base * 1.5, "CONTROL FAILED - the ablation is a no-op. Stop and debug."
print("  control passed: ablation genuinely perturbs the model")

## 4 - How many features can you delete before it matters?

Remove `k` randomly chosen features, re-forecast, restore. Averaged over 3 draws so a
single unlucky choice doesn't drive the number.

In [ ]:
print(f"{'removed':>9}{'% of FFN':>10}{'WQL':>10}{'change':>10}")
curve = []
for frac in (0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0):
    k = int(TOTAL*frac)
    if   k == 0:     s = base
    elif frac == 1.0: s = allz
    else:
        s = float(np.mean([ablate(ffn, n_feat,
                  np.random.default_rng(sd).choice(TOTAL, k, replace=False), score)
                  for sd in range(3)]))
    curve.append((frac, s))
    print(f"{k:>9}{frac*100:>9.0f}%{s:>10.4f}{(s-base)/base*100:>+9.1f}%")

## 5 - Does *any single* feature matter?

Zero one feature at a time -- row `j` of `wi` (what it reads) and column `j` of `wo` (what
it writes). Set `N_SINGLE = 0` for an exhaustive sweep.

In [ ]:
N_SINGLE = 2000        # 0 = every feature

n = TOTAL if N_SINGLE == 0 else min(N_SINGLE, TOTAL)
picks = np.random.default_rng(0).choice(TOTAL, n, replace=False)
t0 = time.time()
d = np.array([ablate(ffn, n_feat, [p], score) - base for p in picks])
print(f"{n} single-feature ablations in {time.time()-t0:.0f}s\n")
print(f"largest DAMAGE : {d.max():+.5f}  ({d.max()/base*100:+.2f}% of baseline)")
print(f"largest BENEFIT: {d.min():+.5f}  ({d.min()/base*100:+.2f}%)")
print(f"removal hurts {(d>0).mean()*100:.0f}% of features, HELPS {(d<0).mean()*100:.0f}%")
a = np.sort(np.abs(d))[::-1]
print(f"concentration  : top 10% of features hold {a[:max(1,len(a)//10)].sum()/np.abs(d).sum()*100:.0f}% of total |effect|")

del pipe, ffn
if DEVICE == "cuda": torch.cuda.empty_cache()

## 6 - The ladder

Now across model sizes. This is the part that answers the scaling question.

`chronos-2` is the current SOTA (encoder-only); the builder finds FFN modules by name
rather than assuming a layout, so it should work -- and the control will catch it if it
doesn't.

In [ ]:
MODELS = LADDER + ["amazon/chronos-2"]    # drop chronos-2 if it errors
N_SINGLE_LADDER = 2000
SEEDS = 3

rows = []
for mid in MODELS:
    t0 = time.time()
    try:
        pipe, ffn, names, n_feat = build(mid, DEVICE)
    except Exception as e:
        print(f"{mid}: SKIPPED ({type(e).__name__}: {e})\n"); continue
    score = scorer(pipe, [x.to(DEVICE) for x in ins], tgt, 64)
    TOT = len(ffn)*n_feat
    b = score(); az = ablate_all(ffn, score)
    print(f"=== {mid}  ({len(ffn)} layers x {n_feat} = {TOT} features) ===")
    print(f"  baseline {b:.4f} | CONTROL {az:.4f} ({(az-b)/b*100:+.0f}%)")
    if az < b*1.5:
        print("  !! control failed, skipping\n"); del pipe, ffn; continue

    r = {"model": mid.split('/')[-1], "n_features": TOT, "baseline": b}
    for frac in (0.25, 0.5, 0.75):
        s = float(np.mean([ablate(ffn, n_feat,
              np.random.default_rng(sd).choice(TOT, int(TOT*frac), replace=False), score)
              for sd in range(SEEDS)]))
        r[f"drop{int(frac*100)}"] = (s-b)/b*100
        print(f"  {frac:>4.0%} removed -> {(s-b)/b*100:+6.1f}%")
    pk = np.random.default_rng(0).choice(TOT, min(N_SINGLE_LADDER,TOT), replace=False)
    dd = np.array([ablate(ffn, n_feat, [p], score) - b for p in pk])
    r["single_max_pct"] = float(dd.max()/b*100); r["helps_pct"] = float((dd<0).mean()*100)
    print(f"  single max {dd.max()/b*100:+.2f}% | {(dd<0).mean()*100:.0f}% improve | {time.time()-t0:.0f}s\n")
    rows.append(r)
    del pipe, ffn
    if DEVICE=="cuda": torch.cuda.empty_cache()
    json.dump(rows, open("chronos_scaling.json","w"), indent=2)

import pandas as pd
df = pd.DataFrame(rows); df

## 7 - The scaling plot

In [ ]:
import matplotlib.pyplot as plt

# removal fraction is an ORDERED quantity -> one hue stepped light->dark, not categorical
RAMP = ["#86b6ef", "#2a78d6", "#184f95"]
COLS = ["drop25", "drop50", "drop75"]
LBL  = {"drop25": "25% removed", "drop50": "50% removed", "drop75": "75% removed"}

short = [m.replace("chronos-bolt-", "") for m in df.model]
x = np.arange(len(df))

fig, ax = plt.subplots(figsize=(8.4, 5.2))
ax.set_facecolor("#fcfcfb"); fig.patch.set_facecolor("#fcfcfb")
ax.axhline(0, color="#9a9a93", lw=1, ls="--", zorder=1)

for col, c in zip(COLS, RAMP):
    ax.plot(x, df[col], marker="o", ms=8, lw=2.2, color=c, zorder=3,
            markeredgecolor="#fcfcfb", markeredgewidth=2)      # 2px surface ring
    ax.annotate(LBL[col], (x[-1], df[col].iloc[-1]), textcoords="offset points",
                xytext=(10, 0), va="center", fontsize=9.5, color="#33332f")
ax.annotate("no loss", (x[0] - 0.06, 0), textcoords="offset points", xytext=(0, -13),
            ha="left", fontsize=8.5, color="#6b6b64")

# identity is the MODEL, not the raw feature count -- label the axis with both
ax.set_xticks(x)
ax.set_xticklabels([f"{m}\n{n//1000}k feat" for m, n in zip(short, df.n_features)],
                   fontsize=9, color="#33332f")
ax.set_xlim(-0.25, len(df) - 1 + 0.75)
ax.set_ylabel("WQL degradation (%)", fontsize=10, color="#54544c")
ax.set_title(f"FFN deletion tolerance across the Chronos-Bolt ladder ({DATASET})",
             fontsize=12.5, color="#33332f", pad=14, loc="left")
ax.grid(True, axis="y", color="#e8e8e3", lw=0.8); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color("#d4d4cd")
ax.tick_params(colors="#6b6b64", labelsize=9)
plt.tight_layout(); plt.savefig("chronos_scaling.png", dpi=190); plt.show()

print("trend across the ladder:")
for col in COLS:
    a, b = df[col].iloc[0], df[col].iloc[-1]
    print(f"  {LBL[col]:<14} {a:+6.1f}%  ->  {b:+6.1f}%   ({'MORE' if b < a else 'LESS'} redundant at scale)")
print("\ncounter-trend -- largest effect of any SINGLE feature:")
for m, v in zip(short, df.single_max_pct):
    print(f"  {m:<7} {v:+.2f}%")

## 8 - The confound check: is it scale, or is it this dataset?

On ETTh1 the `base` model has a **worse** baseline than `tiny` (0.980 vs 0.939). If the
biggest model is simply badly matched to this data, its fragility under ablation might be
about that rather than about scale.

This cell re-runs the ladder on three datasets. Two things to look for:

1. **Does `base` beat `tiny` on baseline WQL anywhere?** If it never does, the ladder
   comparison is not apples-to-apples and the scaling claim is unsafe.
2. **Does the deletion trend hold on all three?** If it only appears on ETTh1, it is a
   dataset effect wearing a scaling costume.

Group ablation only (no single-feature sweep), so this is quick.

In [ ]:
DATASETS = ["ETTh1", "ETTm1", "ETTh2"]
grid = []

for ds in DATASETS:
    i2, t2 = make_windows(load_ett(ds), 512, 64)
    print(f"\n########## {ds}  (naive WQL {naive_wql(i2, t2, 64):.4f}) ##########")
    for mid in LADDER:
        try:
            pipe, ffn, names, n_feat = build(mid, DEVICE)
        except Exception as e:
            print(f"  {mid}: SKIPPED ({type(e).__name__})"); continue
        sc = scorer(pipe, [x.to(DEVICE) for x in i2], t2, 64)
        TOT = len(ffn) * n_feat
        b = sc(); az = ablate_all(ffn, sc)
        ok = az > b * 1.5
        row = {"dataset": ds, "model": mid.split("/")[-1].replace("chronos-bolt-", ""),
               "n_features": TOT, "baseline": b, "control_ok": ok}
        if ok:
            for frac in (0.25, 0.5, 0.75):
                s = float(np.mean([ablate(ffn, n_feat,
                      np.random.default_rng(sd).choice(TOT, int(TOT * frac), replace=False), sc)
                      for sd in range(3)]))
                row[f"drop{int(frac*100)}"] = (s - b) / b * 100
        grid.append(row)
        print(f"  {row['model']:<6} base WQL {b:.4f} | control {'ok' if ok else 'FAILED'}"
              + ("".join(f" | -{f}% -> {row[f'drop{f}']:+6.1f}%" for f in (25, 50, 75)) if ok else ""))
        del pipe, ffn
        if DEVICE == "cuda": torch.cuda.empty_cache()

g = pd.DataFrame(grid)
json.dump(grid, open("chronos_grid.json", "w"), indent=2)
print("\n=== baseline WQL (does base ever beat tiny?) ===")
print(g.pivot(index="model", columns="dataset", values="baseline").round(4))
print("\n=== degradation at 50% removal ===")
print(g.pivot(index="model", columns="dataset", values="drop50").round(1))

## How to read this

**If the lines go DOWN as models get bigger** -- bigger models tolerate deletion better --
then redundancy is not a small-model artifact. It grows with scale, which means it is a
property of how these models store forecasting behaviour. That is the interesting outcome,
and it is what the CPU pilot suggested (`mini` tolerated 50% removal better than `tiny`).

**If the lines go UP** -- bigger models are more fragile -- then the original finding really
was about over-parameterisation at tiny scale. Also worth knowing, and worth reporting
honestly.

**If the control ever fails** (`all-zeroed` under +50%), ignore everything for that model.
Its FFN layout differs and the ablation missed.

### What this can and cannot claim

It **can** say: on this benchmark, with random ablation, a large fraction of FFN features
can be removed from deployed forecasting models with little loss, and here is how that
scales.

It **cannot** say these features are useless. Random deletion is the *conservative* case --
a targeted pruning method would almost certainly remove more. It is also one dataset
domain (electricity transformers) and one model family. `ETTm1` is roughly 30x stricter
than `ETTh1` at the 50% level, so re-run with `DATASET = "ETTm1"` before believing any
single number.

### If the trend holds, the natural next steps

- **GIFT-Eval** (23 datasets, 7 domains) instead of ETT alone -- the standard leaderboard
- **Moirai / TimesFM** -- is this a Chronos property or a TSFM property?
- **Targeted pruning** -- how far can you actually go, and does a pruned model *deploy*
  faster with equal accuracy? That turns a measurement into something people would use.